# RAG-IDEArq — Consultas SPARQL sobre el Grafo Bibliográfico

Queries predefinidas + editor libre sobre el grafo BIBO de IDEArq.

**Grafo por defecto**: `idearq-graph-instances-dedup-v2.nt` (2,408 autores, 16,359 triples)

**Nota**: este notebook **solo lee** el `.nt`. No regenera el grafo.
Para regenerar, ejecuta `rag-graph-ontologia-mapeo.ipynb` + `normalizar-autores-v2.ipynb`.

In [1]:
import os, sys
from pathlib import Path
from collections import Counter

try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from rdflib import Graph, Namespace, URIRef, Literal, RDF
from rdflib.namespace import FOAF, RDFS, XSD
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BIBO = Namespace("http://purl.org/ontology/bibo/")
DC_  = Namespace("http://purl.org/dc/terms/")
SKOS = Namespace("http://www.w3.org/2004/02/skos/core#")
IDEARQ = Namespace("http://idearq.org/resource/")

DATA_DIR = PROJECT_ROOT / "data" / "biblio-graph"
RESULTS_DIR = PROJECT_ROOT / "data" / "results" / "graph-queries"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Selector de versión del grafo ──────────────────────────────────────
GRAPH_VERSION = "v2"  # "original" | "v1" | "v2"

GRAPH_PATHS = {
    "original": DATA_DIR / "idearq-graph-instances.nt",
    "v1":       DATA_DIR / "idearq-graph-instances-dedup.nt",
    "v2":       DATA_DIR / "idearq-graph-instances-dedup-v2.nt",
}

print(f"Cargando grafo '{GRAPH_VERSION}' desde {GRAPH_PATHS[GRAPH_VERSION]} ...")
g = Graph()
g.parse(str(GRAPH_PATHS[GRAPH_VERSION]), format="nt")
print(f"  {len(g):,} triples cargados")

# Estadísticas rápidas
type_counts = Counter()
for obj in set(g.objects(None, RDF.type)):
    type_counts[str(obj).split("/")[-1]] = sum(1 for _ in g.subjects(RDF.type, obj))
print(f"\nDistribución por tipo:")
for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {t:30s} {c:>6,}")

# Helper para imprimir resultados
def run_query(query, title="", max_rows=30):
    if title:
        print(f"\n{'='*60}")
        print(f"  {title}")
        print(f"{'='*60}")
    results = list(g.query(query))
    if not results:
        print("  (sin resultados)")
        return results
    labels = [str(l) for l in results[0].labels] if hasattr(results[0], 'labels') else ["?"]
    print(f"  {' | '.join(labels)}")
    print(f"  {'-'*60}")
    for row in results[:max_rows]:
        vals = [str(v)[:45] for v in row]
        print(f"  {' | '.join(vals)}")
    if len(results) > max_rows:
        print(f"  ... ({len(results) - max_rows} más)")
    print(f"\n  Total: {len(results)} filas")
    return results

Cargando grafo 'v2' desde /home/raglinux/RAG/data/biblio-graph/idearq-graph-instances-dedup-v2.nt ...


  16,359 triples cargados

Distribución por tipo:
  Person                          2,408
  AcademicArticle                 1,062
  Journal                            89
  Book                               57
  Document                           34
  core#Concept                       22
  Series                              4
  Proceedings                         1


## Autores

In [2]:
run_query("""
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name (COUNT(DISTINCT ?article) AS ?n) WHERE {
  ?article dc:creator ?author .
  ?author foaf:name ?name .
}
GROUP BY ?name
HAVING (COUNT(DISTINCT ?article) >= 5)
ORDER BY DESC(?n)
""", "Top 30 autores por nº de publicaciones (>=5)", max_rows=30)


  Top 30 autores por nº de publicaciones (>=5)


  name | n
  ------------------------------------------------------------
  Antonio M. Monge Soares | 16
  Pablo Arias Cabal | 15
  João Luis Cardoso | 15
  Jordà, Guillem Pérez | 14
  R. E. M. HEDGES | 14
  Maestre, Francisco Javier Jover | 13
  Juan F. Gibaja | 13
  Díaz-Zorita Bonilla, Marta | 12
  C. BRONK RAMSEY | 12
  Sanjuán, Leonardo García | 12
  Antonio Faustino Carvalho | 12
  Germán Delibes de Castro | 12
  Jesús Sesma Sesma | 12
  Pena, Rafael Garrido | 11
  Peña-Chocarro, Leonor | 11
  María Pilar Prieto Martínez | 11
  WILLIAM H. WALDREN | 11
  García-Martínez-De-Lagrán, Íñigo | 11
  Víctor M. Guerrero Ayuso | 11
  Alfonso Alday Ruiz | 10
  López-Sáez, José Antonio | 10
  Primitiva Bueno Ramírez | 10
  G. J. VAN KLINKEN | 10
  R. A. HOUSLEY | 10
  José María Rodanés Vicente | 9
  Mujika-Alustiza, José Antonio | 9
  Ángel ESPARZA ARROYO | 9
  Oreto García-Puchol | 9
  Hernández Pérez, Mauro S. | 9
  Aranda Jiménez, Gonzalo | 9
  ... (109 más)

  Total: 139 filas


[(rdflib.term.Literal('Antonio M. Monge Soares'),
  rdflib.term.Literal('16', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Pablo Arias Cabal'),
  rdflib.term.Literal('15', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('João Luis Cardoso'),
  rdflib.term.Literal('15', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Jordà, Guillem Pérez'),
  rdflib.term.Literal('14', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('R. E. M. HEDGES'),
  rdflib.term.Literal('14', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Maestre, Francisco Javier Jover'),
  rdflib.term.Literal('13', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Juan F. Gibaja'),
  rdflib.term.Literal('13', datatype=rdflib.term.URIRef(

In [3]:
# Distribución de publicaciones por década
results = list(g.query("""
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX bibo: <http://purl.org/ontology/bibo/>
SELECT ?year WHERE {
  ?article dc:issued ?year .
  ?article a ?type .
  FILTER(STRSTARTS(STR(?type), STR(bibo:)))
}
ORDER BY ?year
"""))

decades = Counter()
for r in results:
    try:
        year = int(str(r.year)[:4])
        decade = (year // 10) * 10
        decades[decade] += 1
    except (ValueError, IndexError):
        pass

print("Publicaciones por década:")
for d, c in sorted(decades.items()):
    bar = "█" * (c // 2)
    print(f"  {d}s  {c:>4}  {bar}")

fig, ax = plt.subplots(figsize=(10, 4))
years = sorted(decades.keys())
counts = [decades[y] for y in years]
ax.bar(years, counts, color="#4A90D9", edgecolor="white", width=8)
ax.set_xlabel("Década")
ax.set_ylabel("Nº publicaciones")
ax.set_title("Distribución de publicaciones por década")
ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45)
plt.tight_layout()
out = RESULTS_DIR / "pubs_por_decada.png"
plt.savefig(out, dpi=150)
plt.close()
print(f"\nGráfico guardado: {out}")

Publicaciones por década:
  1910s     1  
  1920s     1  
  1930s     1  
  1940s     5  ██
  1950s     5  ██
  1960s    27  █████████████
  1970s    64  ████████████████████████████████
  1980s   107  █████████████████████████████████████████████████████
  1990s   208  ████████████████████████████████████████████████████████████████████████████████████████████████████████
  2000s   276  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  2010s   371  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  2020s    91  █████████████████████████████████████████████



Gráfico guardado: /home/raglinux/RAG/data/results/graph-queries/pubs_por_decada.png


In [ ]:
# Co-autorías de un autor concreto
AUTHOR_NAME = "Pablo Arias Cabal"  

run_query(f"""
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?coauthor (COUNT(DISTINCT ?art) AS ?n) WHERE {{
  ?art dc:creator ?author1, ?author2 .
  ?author1 foaf:name \"{AUTHOR_NAME}\" .
  ?author2 foaf:name ?coauthor .
  FILTER(?author1 != ?author2)
}}
GROUP BY ?coauthor
ORDER BY DESC(?n)
""", f"Co-autorías de '{AUTHOR_NAME}'")


  Co-autorías de 'Pablo Arias Cabal'
  coauthor | n
  ------------------------------------------------------------
  Roberto Ontañón Peredo | 5
  Carlos Pérez Suárez | 3
  Jesús Altuna Etxabe | 3
  Angel Armendáriz Gutiérrez | 3
  Juan José Ibáñez Estévez | 2
  Lydia Zapata Peña | 2
  Urquijo, Jesús Emilio González | 2
  José Alfonso Moure Romanillo | 1
  César González Sainz | 1
  Carmen Mensua Calzado | 1
  Raquel Fernández García | 1
  Esteban Alvarez Fernandez | 1
  María Dolores Garralda Benajes | 1
  Marián Cueto Rapado | 1
  Miguel Ángel Fano Mart??nez | 1
  Juan A. Fernández-Tresguerres Velasco | 1
  Luis César Teira Mayolini | 1

  Total: 17 filas


[(rdflib.term.Literal('Roberto Ontañón Peredo'),
  rdflib.term.Literal('5', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Carlos Pérez Suárez'),
  rdflib.term.Literal('3', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Jesús Altuna Etxabe'),
  rdflib.term.Literal('3', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Angel Armendáriz Gutiérrez'),
  rdflib.term.Literal('3', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Juan José Ibáñez Estévez'),
  rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Lydia Zapata Peña'),
  rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Urquijo, Jesús Emilio González'),
  rdflib.term.Literal('2', datatype=rdflib

## Journals / Revistas

In [5]:
run_query("""
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX bibo: <http://purl.org/ontology/bibo/>
SELECT ?title (COUNT(DISTINCT ?doc) AS ?n) WHERE {
  ?doc dc:isPartOf ?j .
  ?j a bibo:Journal .
  ?j dc:title ?title .
}
GROUP BY ?title
ORDER BY DESC(?n)
LIMIT 20
""", "Top 20 journals por nº de artículos")


  Top 20 journals por nº de artículos
  title | n
  ------------------------------------------------------------
  Trabajos de Prehistoria | 70
  Radiocarbon | 46
  Munibe Antropologia-Arkeologia | 42
  Complutum | 22
  Revista Portuguesa de Arqueologia | 20
  Mitteilungen Des Deutschen Archaologischen In | 18
  Unknown Journal | 17
  Quaternary International | 16
  Spal | 16
  Archivo de Prehistoria Levantina | 12
  Quaderns de Prehistòria I Arqueología de Cast | 12
  Journal of Archaeological Science | 12
  Cuadernos de Prehistoria y Arqueologia de la  | 11
  Zephyrvs-Revista de Prehistoria y Arqueologia | 10
  Salduie. Estudios de Prehistoria y Arqueologí | 8
  Estudios de Arqueología Alavesa | 8
  Espacio, Tiempo y Forma. Prehistoria y Arqueo | 5
  Sagvntvm-Papeles del Laboratorio de Arqueolog | 4
  Lucentum | 4
  Estudios de Prehistoria y Arqueología Madrile | 4

  Total: 20 filas


[(rdflib.term.Literal('Trabajos de Prehistoria'),
  rdflib.term.Literal('70', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Radiocarbon'),
  rdflib.term.Literal('46', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Munibe Antropologia-Arkeologia'),
  rdflib.term.Literal('42', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Complutum'),
  rdflib.term.Literal('22', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Revista Portuguesa de Arqueologia'),
  rdflib.term.Literal('20', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Mitteilungen Des Deutschen Archaologischen Instituts. Abteilung Madrid'),
  rdflib.term.Literal('18', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('Unknown Journal'),
  

In [6]:
# Journals activos por década
results = list(g.query("""
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX bibo: <http://purl.org/ontology/bibo/>
SELECT ?journal ?year WHERE {
  ?doc dc:isPartOf ?j .
  ?j a bibo:Journal .
  ?j dc:title ?journal .
  ?doc dc:issued ?year .
}
"""))

journal_decades = {}
for r in results:
    try:
        year = int(str(r.year)[:4])
        decade = (year // 10) * 10
        journal_decades.setdefault(str(r.journal), set()).add(decade)
    except (ValueError, IndexError):
        pass

print("Journals con más décadas de actividad:")
top_journals = sorted(journal_decades.items(), key=lambda x: -len(x[1]))[:15]
for journal, decs in top_journals:
    dec_str = ", ".join(str(d) + "s" for d in sorted(decs))
    print(f"  {journal:55s}  {len(decs):>2} décadas  {dec_str}")

Journals con más décadas de actividad:
  Radiocarbon                                               7 décadas  1960s, 1970s, 1980s, 1990s, 2000s, 2010s, 2020s
  Archivo de Prehistoria Levantina                          6 décadas  1950s, 1960s, 1980s, 1990s, 2000s, 2010s
  Trabajos de Prehistoria                                   6 décadas  1970s, 1980s, 1990s, 2000s, 2010s, 2020s
  Mitteilungen Des Deutschen Archaologischen Instituts. Abteilung Madrid   6 décadas  1960s, 1970s, 1980s, 1990s, 2000s, 2010s
  Munibe Antropologia-Arkeologia                            5 décadas  1980s, 1990s, 2000s, 2010s, 2020s
  Cuadernos de Prehistoria y Arqueologia de la Universidad Autonoma de Madrid   5 décadas  1970s, 1980s, 1990s, 2010s, 2020s
  Spal                                                      4 décadas  1990s, 2000s, 2010s, 2020s
  Lucentum                                                  4 décadas  1980s, 2000s, 2010s, 2020s
  Complutum                                                 4 déc

## Documentos

In [7]:
run_query("""
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX bibo: <http://purl.org/ontology/bibo/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?type (COUNT(?s) AS ?n) WHERE {
  ?s rdf:type ?type .
  FILTER(STRSTARTS(STR(?type), STR(bibo:)))
}
GROUP BY ?type
ORDER BY DESC(?n)
""", "Distribución de tipos de documento BIBO")


  Distribución de tipos de documento BIBO


  type | n
  ------------------------------------------------------------
  http://purl.org/ontology/bibo/AcademicArticle | 1062
  http://purl.org/ontology/bibo/Journal | 89
  http://purl.org/ontology/bibo/Book | 57
  http://purl.org/ontology/bibo/Document | 34
  http://purl.org/ontology/bibo/Series | 4
  http://purl.org/ontology/bibo/Proceedings | 1

  Total: 6 filas


[(rdflib.term.URIRef('http://purl.org/ontology/bibo/AcademicArticle'),
  rdflib.term.Literal('1062', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://purl.org/ontology/bibo/Journal'),
  rdflib.term.Literal('89', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://purl.org/ontology/bibo/Book'),
  rdflib.term.Literal('57', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://purl.org/ontology/bibo/Document'),
  rdflib.term.Literal('34', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://purl.org/ontology/bibo/Series'),
  rdflib.term.Literal('4', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.URIRef('http://purl.org/ontology/bibo/Proceedings'),
  rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'

In [8]:
# Artículos académicos SIN DOI
run_query("""
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX bibo: <http://purl.org/ontology/bibo/>
SELECT ?title ?year WHERE {
  ?art a bibo:AcademicArticle .
  ?art dc:title ?title .
  OPTIONAL { ?art dc:issued ?year }
  FILTER NOT EXISTS { ?art bibo:doi ?doi }
}
ORDER BY ?year
LIMIT 20
""", "Artículos académicos sin DOI (primeros 20)")


  Artículos académicos sin DOI (primeros 20)


  title | year
  ------------------------------------------------------------
  La Cueva de las Campanas (Gualchos, Granada): | None
  El arte rupestre en España | 1915
  Las pinturas rupestres de los alrededores de  | 1927
  El Cau den Serra (Cueva Sepulcral de Picamoix | 1940
  Un nuevo grupo de pinturas rupestres en Albar | 1949
  Las pinturas rupestres del Barranco de Les Do | 1953
  La Cueva de Bricia (Asturias) | 1954
  Las Necrópolis de Ampurias | 1955
  El Castillarejo de los Moros (Andilla, Valenc | 1958
  Ergebnisse einer Ersten Stratigraphischen Unt | 1962
  Resultado del Análisis Polínico de una Serie  | 1963
  Hallazgos Prehistóricos en Les Llometes (Alco | 1964
  Los materiales encontrados en la Cueva de Son | 1966
  El yacimiento de Myotragus balearicus en las  | 1966
  Hallazgos Antropológicos de la Cueva del Aer | 1966
  Mallorca chronology for Prehistory based on r | 1967
  Cuevas Sepulcrales de Lechón, Arralday, Calav | 1967
  Toscanos und Trayamar | 1968
  La Dataci

[(rdflib.term.Literal('La Cueva de las Campanas (Gualchos, Granada): Un Yacimiento Neolítico en la Costa Granadina'),
  None),
 (rdflib.term.Literal('El arte rupestre en España'),
  rdflib.term.Literal('1915', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#gYear'))),
 (rdflib.term.Literal('Las pinturas rupestres de los alrededores de Tormón'),
  rdflib.term.Literal('1927', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#gYear'))),
 (rdflib.term.Literal('El Cau den Serra (Cueva Sepulcral de Picamoixons, Término de Valls)'),
  rdflib.term.Literal('1940', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#gYear'))),
 (rdflib.term.Literal('Un nuevo grupo de pinturas rupestres en Albarracín: La Cueva de Doña Clotilde'),
  rdflib.term.Literal('1949', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#gYear'))),
 (rdflib.term.Literal('Las pinturas rupestres del Barranco de Les Dogues'),
  rdflib.term.Literal('1953', datatype=rdflib.term.URIRe

In [9]:
# Artículos sin ISSN / sin journal
run_query("""
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX bibo: <http://purl.org/ontology/bibo/>
SELECT ?title WHERE {
  ?art a bibo:AcademicArticle .
  ?art dc:title ?title .
  FILTER NOT EXISTS { ?art dc:isPartOf ?j }
}
LIMIT 15
""", "Artículos sin journal (dc:isPartOf)")


  Artículos sin journal (dc:isPartOf)
  title
  ------------------------------------------------------------
  Cronología del Arte Rupestre Postpaleolítico 
  El Poblado de El Castañuelo (Aracena) y el Po
  Combining Small-Vertebrate, Marine and Stable
  Los Castellazos (Mediana de Aragón, Zaragoza)
  La Necrópolis Ibérica de El Cigarralejo (Mula
  El Cerro de la Campana y su Cronología según 
  Aproximación a la arqueología funeraria de la
  La Cueva de Cofresnedo: Actuaciones 2000 2001
  La necrópolis protohistórica de Cales Coves, 
  La Campagne de Fouilles 1981 à Castellar (Jae
  Dólmenes de La Lora (Burgos)
  Bone Needles in Mallorcan Prehistory: a Reapp
  Towards the Periodization of the Uses of Can 
  Industria Ósea y Funcionalidad: Neolítico y C
  El Abrigo de El Esplugón (Billobas-Sabiñánigo

  Total: 15 filas


[(rdflib.term.Literal('Cronología del Arte Rupestre Postpaleolítico y Datación Absoluta de Pátinas de Oxalato Cálcico: Primeras Experiencias en Castilla-La Mancha (2004-2007)'),),
 (rdflib.term.Literal('El Poblado de El Castañuelo (Aracena) y el Post-Orientalizante en la Sierra Norte de Huelva'),),
 (rdflib.term.Literal('Combining Small-Vertebrate, Marine and Stable-Isotope Data to Reconstruct Past Environments'),),
 (rdflib.term.Literal('Los Castellazos (Mediana de Aragón, Zaragoza)'),),
 (rdflib.term.Literal('La Necrópolis Ibérica de El Cigarralejo (Mula, Murcia)'),),
 (rdflib.term.Literal('El Cerro de la Campana y su Cronología según el C-14 (Yecla, Murcia)'),),
 (rdflib.term.Literal('Aproximación a la arqueología funeraria de las culturas iniciales de la Prehistoria de Mallorca'),),
 (rdflib.term.Literal('La Cueva de Cofresnedo: Actuaciones 2000 2001'),),
 (rdflib.term.Literal('La necrópolis protohistórica de Cales Coves, Menorca'),),
 (rdflib.term.Literal('La Campagne de Fouilles 

In [10]:
# Artículos sin abstract
run_query("""
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX bibo: <http://purl.org/ontology/bibo/>
SELECT ?title WHERE {
  ?art a bibo:AcademicArticle .
  ?art dc:title ?title .
  FILTER NOT EXISTS { ?art bibo:abstract ?abs }
}
LIMIT 15
""", "Artículos sin abstract (primeros 15)")


  Artículos sin abstract (primeros 15)
  title
  ------------------------------------------------------------
  Cronología del Arte Rupestre Postpaleolítico 
  El Poblado de El Castañuelo (Aracena) y el Po
  Damaged Burials or reliquiae Cogotenses? On t
  Dental Morphology: A Valuable Contribution to
  Los Castellazos (Mediana de Aragón, Zaragoza)
  La Necrópolis Ibérica de El Cigarralejo (Mula
  El Cerro de la Campana y su Cronología según 
  Completando el Mapa de la Cuenca del Ebro: El
  La Cueva de Cofresnedo: Actuaciones 2000 2001
  La necrópolis protohistórica de Cales Coves, 
  La Campagne de Fouilles 1981 à Castellar (Jae
  Dólmenes de La Lora (Burgos)
  Industria Ósea y Funcionalidad: Neolítico y C
  El Abrigo de El Esplugón (Billobas-Sabiñánigo
  La Cultura" de El Argar"

  Total: 15 filas


[(rdflib.term.Literal('Cronología del Arte Rupestre Postpaleolítico y Datación Absoluta de Pátinas de Oxalato Cálcico: Primeras Experiencias en Castilla-La Mancha (2004-2007)'),),
 (rdflib.term.Literal('El Poblado de El Castañuelo (Aracena) y el Post-Orientalizante en la Sierra Norte de Huelva'),),
 (rdflib.term.Literal('Damaged Burials or reliquiae Cogotenses? On the Accompanying Human Bones in Burial Pits Belonging to the Iberian Bronze Age'),),
 (rdflib.term.Literal('Dental Morphology: A Valuable Contribution to Our Understanding of Prehistory'),),
 (rdflib.term.Literal('Los Castellazos (Mediana de Aragón, Zaragoza)'),),
 (rdflib.term.Literal('La Necrópolis Ibérica de El Cigarralejo (Mula, Murcia)'),),
 (rdflib.term.Literal('El Cerro de la Campana y su Cronología según el C-14 (Yecla, Murcia)'),),
 (rdflib.term.Literal('Completando el Mapa de la Cuenca del Ebro: El Mesolítico del IX Milenio Cal BP de Espantalobos (Huesca, España)'),),
 (rdflib.term.Literal('La Cueva de Cofresnedo: A

## Editor SPARQL libre

Edita la variable `query` y ejecuta la celda. Los resultados se muestran en tabla.

In [11]:
query = """
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX bibo: <http://purl.org/ontology/bibo/>

# Ejemplo: artículos de un autor en un journal concreto
SELECT ?title ?year ?journal WHERE {
  ?art dc:creator ?author .
  ?author foaf:name ?authorName .
  ?art dc:title ?title .
  OPTIONAL { ?art dc:issued ?year }
  OPTIONAL {
    ?art dc:isPartOf ?j .
    ?j dc:title ?journal .
  }
  FILTER(CONTAINS(LCASE(?authorName), "ariás") || CONTAINS(LCASE(?authorName), "arias"))
}
ORDER BY ?year
LIMIT 20
"""

print(f"Ejecutando query:\n{query}\n")
try:
    results = list(g.query(query))
    if not results:
        print("(sin resultados)")
    else:
        labels = [str(l) for l in results[0].labels] if hasattr(results[0], 'labels') else ["?"]
        print(f"  {' | '.join(labels)}")
        print(f"  {'-'*80}")
        for row in results:
            vals = [str(v)[:40] for v in row]
            print(f"  {' | '.join(vals)}")
        print(f"\n  Total: {len(results)} filas")
except Exception as e:
    print(f"Error en la query: {e}")

Ejecutando query:

PREFIX dc: <http://purl.org/dc/terms/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX bibo: <http://purl.org/ontology/bibo/>

# Ejemplo: artículos de un autor en un journal concreto
SELECT ?title ?year ?journal WHERE {
  ?art dc:creator ?author .
  ?author foaf:name ?authorName .
  ?art dc:title ?title .
  OPTIONAL { ?art dc:issued ?year }
  OPTIONAL {
    ?art dc:isPartOf ?j .
    ?j dc:title ?journal .
  }
  FILTER(CONTAINS(LCASE(?authorName), "ariás") || CONTAINS(LCASE(?authorName), "arias"))
}
ORDER BY ?year
LIMIT 20




  title | year | journal
  --------------------------------------------------------------------------------
  Investigaciones Prehistóricas en la Sier | 1990 | None
  Las Sepulturas de la Cueva de los Canes  | 1990 | Trabajos de Prehistoria
  De Cazadores a Campesinos: La Transición | 1991 | None
  Estrategias Económicas de las Poblacione | 1992 | None
  Las Excavaciones Arqueológicas de la Cue | 1992 | None
  El Neolítico de la Región Cantábrica: Nu | 1994 | None
  Nuevas Dataciones Absolutas para el Neol | 1999 | Munibe Antropologia-Arkeologia
  Nuevas Aportaciones al Conocimiento de l | 1999 | None
  Excavaciones Arqueológicas en la Cueva d | 1999 | None
  Sondeos Arqueológicos en Yacimientos en  | 2000 | None
  La Transición al Neolítico en la Región  | 2000 | None
  Estudio Integral del Complejo Arqueológi | 2000 | None
  Excavación Arqueológica de Urgencia en l | 2007 | None
  A View from the Edges: The Mesolithic Se | 2009 | None
  La Chronologie du Néolithique Ancien Car | 2010

## Estadísticas globales del grafo

In [12]:
# Estadísticas globales (queries individuales para evitar problemas de subqueries)
print(f"Grafo: {GRAPH_VERSION} | {len(g):,} triples\n")

stats = {}
stats['Autores'] = len(set(g.objects(None, FOAF.name)))
stats['AcademicArticle'] = sum(1 for _ in g.subjects(RDF.type, BIBO.AcademicArticle))
stats['Book'] = sum(1 for _ in g.subjects(RDF.type, BIBO.Book))
stats['Document'] = sum(1 for _ in g.subjects(RDF.type, BIBO.Document))
stats['Journals'] = sum(1 for _ in g.subjects(RDF.type, BIBO.Journal))
stats['Con DOI'] = sum(1 for _ in g.subjects(BIBO.doi, None))
stats['Con ISSN'] = sum(1 for _ in g.subjects(BIBO.issn, None))
stats['Con abstract'] = sum(1 for _ in g.subjects(BIBO.abstract, None))
stats['Temas'] = sum(1 for _ in g.subjects(RDF.type, SKOS.Concept))

print(f"  {'Métrica':<25s}  {'Valor':>8}")
print(f"  {'-'*35}")
for k, v in stats.items():
    print(f"  {k:<25s}  {v:>8,}")

total_articles = stats['AcademicArticle'] + stats['Book'] + stats['Document']
print(f"\n  Cobertura de metadatos:")
print(f"    DOI:      {stats['Con DOI']/total_articles*100:.1f}%")
print(f"    ISSN:     {stats['Con ISSN']/total_articles*100:.1f}%")
print(f"    Abstract: {stats['Con abstract']/total_articles*100:.1f}%")


Grafo: v2 | 16,359 triples

  Métrica                       Valor
  -----------------------------------
  Autores                       2,408
  AcademicArticle               1,062
  Book                             57
  Document                         34
  Journals                         89
  Con DOI                         493
  Con ISSN                        552
  Con abstract                    547
  Temas                            22

  Cobertura de metadatos:
    DOI:      42.8%
    ISSN:     47.9%
    Abstract: 47.4%
